In [39]:
import random
import numpy as np
import pandas as pd

In [40]:
# 正常抽獎
def normal_draw():
    r = random.random() # 0 ~ 1
    if r < 0.05:
        return '頭獎'
    elif r < 0.15:
        return '貳獎'
    else:
        return '未中獎'

In [41]:
# 保底抽獎
def guaranteed_draw():
    r = random.random()
    if r < 1/3:
        return '頭獎' # 0.33 中頭獎
    else:
        return '貳獎' # 0.66 中貳獎

In [42]:
# 帳號狀態清單 可更動
accounts = {
    '611211101':{'is_new_player': True, "new_draws": 0, "new_has_won": False, "lose_conti": 0},
    '611211102':{'is_new_player': True, "new_draws": 2, "new_has_won": False, "lose_conti": 0},
    '611211104':{'is_new_player': True, "new_draws": 3, "new_has_won": True, "lose_conti": 0},
    '611211105':{'is_new_player': True, "new_draws": 2, "new_has_won": True, "lose_conti": 0},
}

# 遊戲機制
def play_game(account, times): # (帳號, 遊玩次數)
    
    if account not in accounts:
        # 新手期：前 5 期至少中 1 次
        accounts[account] = {
            'is_new_player': True,
            'new_draws': 0, # 新手期已抽幾次
            'new_has_won': False, # 新手期是否已中過
            'lose_conti': 0 # 非新手期連續未中次數
        }
        print(f'帳號：{account}，新建立玩家資料')
    else:
        print(f'帳號：{account}，已存在玩家資料')

    player = accounts[account]
    
    if player['new_draws'] >= 5:
        player['is_new_player'] = False

    # 帳號狀態確認
    if player['is_new_player']:
        if player['new_has_won']:
            print('目前狀態：新手期，但您已中獎，不會再觸發新手保底')
        else:
            print('目前狀態：新手期，尚未中獎，仍具有新手保底')
    else:
        print('目前狀態：非新手期，您不具有新手保底')

    print(f'開始遊玩{times}次')

    rewards = [] # 紀錄每次獎金
    
    for i in range(1, times+1):
        trigger = '正常抽獎'
        mes = False
        
        # 新手期 前 5 抽至少中 1
        if player['is_new_player'] and player['new_draws'] < 5:
            if player['new_draws'] == 4 and player['new_has_won'] == False:
                result = guaranteed_draw()
                trigger = '新手保底'
            else:
                result = normal_draw()

            player['new_draws'] = player['new_draws'] + 1

            if result != '未中獎':
                if player['new_has_won'] == False:
                    mes = True
                player['new_has_won'] = True
                player['is_new_player'] = False

            elif player['new_draws'] == 5:
                player['is_new_player'] = False
        
        # 非新手期 20 抽至少中 1
        else:
            if player['lose_conti'] >= 19:
                result = guaranteed_draw()
                trigger = '20抽大保'
            else:
                result = normal_draw()

            if result == '未中獎':
                player['lose_conti'] = player['lose_conti'] + 1
            else:
                player['lose_conti'] = 0

        if result == '頭獎':
            reward = 500
            text = '頭獎 500 元'
        elif result == '貳獎':
            reward = 200
            text = '貳獎 200元'
        else:
            reward = 0
            text = '未中獎 再接再厲'

        rewards.append(reward)

        print(f'第{i}次：{text}，觸發{trigger}')

        if mes:
            print('您已不是新手期，之後不具有新手保底')
            
    print('遊玩結束後玩家狀態：')
    print(player)

    rewards = np.array(rewards)
    n = len(rewards)

    x_bar = rewards.mean()
    s = rewards.std(ddof=1)
    se = s / (n**(1/2))

    z = 1.96
    CI_UPPER = x_bar + z * se
    CI_LOWER = x_bar - z * se
    
    summary = pd.DataFrame(
        {
            '樣本數': [n],
            '單次遊玩獎金期望值估計':[f'{x_bar:.2f}'],
            '標準誤': [f'{se:.2f}'],
            'CI下界': [round(CI_LOWER, 2)],
            'CI上界': [round(CI_UPPER, 2)]
        }
    )
    return summary

In [43]:
while True:
    account = input('請輸入帳號（輸入 /c 離開）：')
    
    if account == '/c':
        print('程式結束')
        break

    summary = play_game(account, times=5000)
    display(summary)

帳號：1，新建立玩家資料
目前狀態：新手期，尚未中獎，仍具有新手保底
開始遊玩5000次
第1次：未中獎 再接再厲，觸發正常抽獎
第2次：未中獎 再接再厲，觸發正常抽獎
第3次：未中獎 再接再厲，觸發正常抽獎
第4次：未中獎 再接再厲，觸發正常抽獎
第5次：貳獎 200元，觸發新手保底
您已不是新手期，之後不具有新手保底
第6次：未中獎 再接再厲，觸發正常抽獎
第7次：未中獎 再接再厲，觸發正常抽獎
第8次：未中獎 再接再厲，觸發正常抽獎
第9次：貳獎 200元，觸發正常抽獎
第10次：未中獎 再接再厲，觸發正常抽獎
第11次：未中獎 再接再厲，觸發正常抽獎
第12次：未中獎 再接再厲，觸發正常抽獎
第13次：貳獎 200元，觸發正常抽獎
第14次：未中獎 再接再厲，觸發正常抽獎
第15次：未中獎 再接再厲，觸發正常抽獎
第16次：未中獎 再接再厲，觸發正常抽獎
第17次：未中獎 再接再厲，觸發正常抽獎
第18次：未中獎 再接再厲，觸發正常抽獎
第19次：未中獎 再接再厲，觸發正常抽獎
第20次：未中獎 再接再厲，觸發正常抽獎
第21次：未中獎 再接再厲，觸發正常抽獎
第22次：頭獎 500 元，觸發正常抽獎
第23次：未中獎 再接再厲，觸發正常抽獎
第24次：未中獎 再接再厲，觸發正常抽獎
第25次：未中獎 再接再厲，觸發正常抽獎
第26次：未中獎 再接再厲，觸發正常抽獎
第27次：未中獎 再接再厲，觸發正常抽獎
第28次：未中獎 再接再厲，觸發正常抽獎
第29次：未中獎 再接再厲，觸發正常抽獎
第30次：未中獎 再接再厲，觸發正常抽獎
第31次：頭獎 500 元，觸發正常抽獎
第32次：未中獎 再接再厲，觸發正常抽獎
第33次：未中獎 再接再厲，觸發正常抽獎
第34次：未中獎 再接再厲，觸發正常抽獎
第35次：未中獎 再接再厲，觸發正常抽獎
第36次：未中獎 再接再厲，觸發正常抽獎
第37次：未中獎 再接再厲，觸發正常抽獎
第38次：未中獎 再接再厲，觸發正常抽獎
第39次：未中獎 再接再厲，觸發正常抽獎
第40次：未中獎 再接再厲，觸發正常抽獎
第41次：未中獎 再接再厲，觸發正常抽獎
第42次：貳獎 200元，觸發正常抽獎
第43次：未中獎 再接再厲，觸發正常抽獎
第44次：未中獎 再接再厲，觸發正常抽獎
第45次：未中獎 再接再厲，觸發正常抽獎
第46次：

,樣本數,單次遊玩獎金期望值估計,標準誤,CI下界,CI上界
0,5000,46.84,1.72,43.46,50.22


帳號：2，新建立玩家資料
目前狀態：新手期，尚未中獎，仍具有新手保底
開始遊玩5000次
第1次：未中獎 再接再厲，觸發正常抽獎
第2次：未中獎 再接再厲，觸發正常抽獎
第3次：未中獎 再接再厲，觸發正常抽獎
第4次：貳獎 200元，觸發正常抽獎
您已不是新手期，之後不具有新手保底
第5次：未中獎 再接再厲，觸發正常抽獎
第6次：未中獎 再接再厲，觸發正常抽獎
第7次：未中獎 再接再厲，觸發正常抽獎
第8次：未中獎 再接再厲，觸發正常抽獎
第9次：未中獎 再接再厲，觸發正常抽獎
第10次：未中獎 再接再厲，觸發正常抽獎
第11次：未中獎 再接再厲，觸發正常抽獎
第12次：貳獎 200元，觸發正常抽獎
第13次：貳獎 200元，觸發正常抽獎
第14次：未中獎 再接再厲，觸發正常抽獎
第15次：未中獎 再接再厲，觸發正常抽獎
第16次：未中獎 再接再厲，觸發正常抽獎
第17次：頭獎 500 元，觸發正常抽獎
第18次：未中獎 再接再厲，觸發正常抽獎
第19次：未中獎 再接再厲，觸發正常抽獎
第20次：未中獎 再接再厲，觸發正常抽獎
第21次：頭獎 500 元，觸發正常抽獎
第22次：未中獎 再接再厲，觸發正常抽獎
第23次：未中獎 再接再厲，觸發正常抽獎
第24次：未中獎 再接再厲，觸發正常抽獎
第25次：未中獎 再接再厲，觸發正常抽獎
第26次：未中獎 再接再厲，觸發正常抽獎
第27次：未中獎 再接再厲，觸發正常抽獎
第28次：未中獎 再接再厲，觸發正常抽獎
第29次：未中獎 再接再厲，觸發正常抽獎
第30次：未中獎 再接再厲，觸發正常抽獎
第31次：未中獎 再接再厲，觸發正常抽獎
第32次：貳獎 200元，觸發正常抽獎
第33次：貳獎 200元，觸發正常抽獎
第34次：未中獎 再接再厲，觸發正常抽獎
第35次：未中獎 再接再厲，觸發正常抽獎
第36次：未中獎 再接再厲，觸發正常抽獎
第37次：未中獎 再接再厲，觸發正常抽獎
第38次：未中獎 再接再厲，觸發正常抽獎
第39次：未中獎 再接再厲，觸發正常抽獎
第40次：未中獎 再接再厲，觸發正常抽獎
第41次：貳獎 200元，觸發正常抽獎
第42次：未中獎 再接再厲，觸發正常抽獎
第43次：未中獎 再接再厲，觸發正常抽獎
第44次：未中獎 再接再厲，觸發正常抽獎
第45次：貳獎 200元，觸發正常抽獎
第46次：未中獎

,樣本數,單次遊玩獎金期望值估計,標準誤,CI下界,CI上界
0,5000,46.94,1.71,43.6,50.28


帳號：3，新建立玩家資料
目前狀態：新手期，尚未中獎，仍具有新手保底
開始遊玩5000次
第1次：貳獎 200元，觸發正常抽獎
您已不是新手期，之後不具有新手保底
第2次：未中獎 再接再厲，觸發正常抽獎
第3次：未中獎 再接再厲，觸發正常抽獎
第4次：未中獎 再接再厲，觸發正常抽獎
第5次：頭獎 500 元，觸發正常抽獎
第6次：未中獎 再接再厲，觸發正常抽獎
第7次：未中獎 再接再厲，觸發正常抽獎
第8次：未中獎 再接再厲，觸發正常抽獎
第9次：頭獎 500 元，觸發正常抽獎
第10次：貳獎 200元，觸發正常抽獎
第11次：未中獎 再接再厲，觸發正常抽獎
第12次：未中獎 再接再厲，觸發正常抽獎
第13次：未中獎 再接再厲，觸發正常抽獎
第14次：未中獎 再接再厲，觸發正常抽獎
第15次：貳獎 200元，觸發正常抽獎
第16次：未中獎 再接再厲，觸發正常抽獎
第17次：未中獎 再接再厲，觸發正常抽獎
第18次：未中獎 再接再厲，觸發正常抽獎
第19次：未中獎 再接再厲，觸發正常抽獎
第20次：未中獎 再接再厲，觸發正常抽獎
第21次：未中獎 再接再厲，觸發正常抽獎
第22次：未中獎 再接再厲，觸發正常抽獎
第23次：未中獎 再接再厲，觸發正常抽獎
第24次：未中獎 再接再厲，觸發正常抽獎
第25次：貳獎 200元，觸發正常抽獎
第26次：未中獎 再接再厲，觸發正常抽獎
第27次：未中獎 再接再厲，觸發正常抽獎
第28次：未中獎 再接再厲，觸發正常抽獎
第29次：未中獎 再接再厲，觸發正常抽獎
第30次：未中獎 再接再厲，觸發正常抽獎
第31次：未中獎 再接再厲，觸發正常抽獎
第32次：未中獎 再接再厲，觸發正常抽獎
第33次：未中獎 再接再厲，觸發正常抽獎
第34次：未中獎 再接再厲，觸發正常抽獎
第35次：未中獎 再接再厲，觸發正常抽獎
第36次：未中獎 再接再厲，觸發正常抽獎
第37次：未中獎 再接再厲，觸發正常抽獎
第38次：未中獎 再接再厲，觸發正常抽獎
第39次：貳獎 200元，觸發正常抽獎
第40次：未中獎 再接再厲，觸發正常抽獎
第41次：頭獎 500 元，觸發正常抽獎
第42次：未中獎 再接再厲，觸發正常抽獎
第43次：未中獎 再接再厲，觸發正常抽獎
第44次：未中獎 再接再厲，觸發正常抽獎
第45次：未中獎 再接再厲，觸發正常抽獎
第46次：未

...

帳號：1，已存在玩家資料
目前狀態：非新手期，您不具有新手保底
開始遊玩5000次
第1次：未中獎 再接再厲，觸發正常抽獎
第2次：未中獎 再接再厲，觸發正常抽獎
第3次：未中獎 再接再厲，觸發正常抽獎
第4次：未中獎 再接再厲，觸發正常抽獎
第5次：未中獎 再接再厲，觸發正常抽獎
第6次：未中獎 再接再厲，觸發正常抽獎
第7次：未中獎 再接再厲，觸發正常抽獎
第8次：未中獎 再接再厲，觸發正常抽獎
第9次：未中獎 再接再厲，觸發正常抽獎
第10次：未中獎 再接再厲，觸發正常抽獎
第11次：貳獎 200元，觸發正常抽獎
第12次：未中獎 再接再厲，觸發正常抽獎
第13次：貳獎 200元，觸發正常抽獎
第14次：頭獎 500 元，觸發正常抽獎
第15次：貳獎 200元，觸發正常抽獎
第16次：未中獎 再接再厲，觸發正常抽獎
第17次：未中獎 再接再厲，觸發正常抽獎
第18次：未中獎 再接再厲，觸發正常抽獎
第19次：未中獎 再接再厲，觸發正常抽獎
第20次：未中獎 再接再厲，觸發正常抽獎
第21次：未中獎 再接再厲，觸發正常抽獎
第22次：未中獎 再接再厲，觸發正常抽獎
第23次：未中獎 再接再厲，觸發正常抽獎
第24次：未中獎 再接再厲，觸發正常抽獎
第25次：未中獎 再接再厲，觸發正常抽獎
第26次：未中獎 再接再厲，觸發正常抽獎
第27次：未中獎 再接再厲，觸發正常抽獎
第28次：未中獎 再接再厲，觸發正常抽獎
第29次：未中獎 再接再厲，觸發正常抽獎
第30次：未中獎 再接再厲，觸發正常抽獎
第31次：貳獎 200元，觸發正常抽獎
第32次：未中獎 再接再厲，觸發正常抽獎
第33次：未中獎 再接再厲，觸發正常抽獎
第34次：貳獎 200元，觸發正常抽獎
第35次：未中獎 再接再厲，觸發正常抽獎
第36次：未中獎 再接再厲，觸發正常抽獎
第37次：未中獎 再接再厲，觸發正常抽獎
第38次：未中獎 再接再厲，觸發正常抽獎
第39次：未中獎 再接再厲，觸發正常抽獎
第40次：未中獎 再接再厲，觸發正常抽獎
第41次：未中獎 再接再厲，觸發正常抽獎
第42次：未中獎 再接再厲，觸發正常抽獎
第43次：未中獎 再接再厲，觸發正常抽獎
第44次：貳獎 200元，觸發正常抽獎
第45次：未中獎 再接再厲，觸發正常抽獎
第46次：未中獎 再接再厲，觸發正常抽獎
第47次：未中

,樣本數,單次遊玩獎金期望值估計,標準誤,CI下界,CI上界
0,5000,44.28,1.68,41.0,47.56


帳號：1，已存在玩家資料
目前狀態：非新手期，您不具有新手保底
開始遊玩5000次
第1次：未中獎 再接再厲，觸發正常抽獎
第2次：未中獎 再接再厲，觸發正常抽獎
第3次：未中獎 再接再厲，觸發正常抽獎
第4次：未中獎 再接再厲，觸發正常抽獎
第5次：貳獎 200元，觸發正常抽獎
第6次：未中獎 再接再厲，觸發正常抽獎
第7次：未中獎 再接再厲，觸發正常抽獎
第8次：未中獎 再接再厲，觸發正常抽獎
第9次：貳獎 200元，觸發正常抽獎
第10次：未中獎 再接再厲，觸發正常抽獎
第11次：未中獎 再接再厲，觸發正常抽獎
第12次：未中獎 再接再厲，觸發正常抽獎
第13次：未中獎 再接再厲，觸發正常抽獎
第14次：貳獎 200元，觸發正常抽獎
第15次：未中獎 再接再厲，觸發正常抽獎
第16次：未中獎 再接再厲，觸發正常抽獎
第17次：未中獎 再接再厲，觸發正常抽獎
第18次：未中獎 再接再厲，觸發正常抽獎
第19次：頭獎 500 元，觸發正常抽獎
第20次：未中獎 再接再厲，觸發正常抽獎
第21次：未中獎 再接再厲，觸發正常抽獎
第22次：頭獎 500 元，觸發正常抽獎
第23次：未中獎 再接再厲，觸發正常抽獎
第24次：未中獎 再接再厲，觸發正常抽獎
第25次：未中獎 再接再厲，觸發正常抽獎
第26次：未中獎 再接再厲，觸發正常抽獎
第27次：未中獎 再接再厲，觸發正常抽獎
第28次：未中獎 再接再厲，觸發正常抽獎
第29次：未中獎 再接再厲，觸發正常抽獎
第30次：貳獎 200元，觸發正常抽獎
第31次：未中獎 再接再厲，觸發正常抽獎
第32次：未中獎 再接再厲，觸發正常抽獎
第33次：未中獎 再接再厲，觸發正常抽獎
第34次：未中獎 再接再厲，觸發正常抽獎
第35次：未中獎 再接再厲，觸發正常抽獎
第36次：未中獎 再接再厲，觸發正常抽獎
第37次：未中獎 再接再厲，觸發正常抽獎
第38次：未中獎 再接再厲，觸發正常抽獎
第39次：未中獎 再接再厲，觸發正常抽獎
第40次：未中獎 再接再厲，觸發正常抽獎
第41次：未中獎 再接再厲，觸發正常抽獎
第42次：未中獎 再接再厲，觸發正常抽獎
第43次：未中獎 再接再厲，觸發正常抽獎
第44次：未中獎 再接再厲，觸發正常抽獎
第45次：貳獎 200元，觸發正常抽獎
第46次：未中獎 再接再厲，觸發正常抽獎
第47次：未

,樣本數,單次遊玩獎金期望值估計,標準誤,CI下界,CI上界
0,5000,45.94,1.71,42.58,49.3


程式結束
